# Fink/LSST — Reload Cepheid/Pulsator Light Curves & Analysis

This notebook reloads the data saved by `02_cepheids_extended_search.ipynb`
from the `data_CEPHEIDS_DDF_02/` directory and reproduces the full analysis
(raw light curves, Lomb-Scargle period search, phase-folded diagrams,
Period-Luminosity diagram) **without any Fink API call**.

### Expected directory layout
```
data_CEPHEIDS_DDF_02/
├── df_obj.parquet
├── df_pulsators.parquet
├── df_periods.parquet        (optional — recomputed below if absent)
├── lc_dict_meta.parquet
└── lightcurves/
    └── {diaObjectId}/
        ├── sources.parquet
        └── fp.parquet
```


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- created : 2026-06-17
- last update : 2026-06-20 ported Lomb-Scargle analysis from 02_cepheids_extended_search.ipynb; flux_col confirmed as r:scienceFlux (NOT r:psfFlux) throughout

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import pathlib
import warnings
import os

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from astropy.timeseries import LombScargle, LombScargleMultiband
from astropy.time import Time
from IPython.display import display

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# to enlarge the sizes
params = {
    "legend.fontsize": "large",
    "figure.figsize": (10, 6),
    "axes.labelsize": "large",
    "axes.titlesize": "large",
    "xtick.labelsize": "large",
    "ytick.labelsize": "large",
}
plt.rcParams.update(params)

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("no ipympl → %matplotlib inline")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "CEPHEIDS_DDF_02"
DIR_DATA = pathlib.Path(f"data_{NB_TAG}")
LC_DIR = DIR_DATA / "lightcurves"

NB_TAG_03 = "CEPHEIDS_DDF_03"
DIR_FIGS = pathlib.Path(f"figs_{NB_TAG_03}")
DIR_FIGS.mkdir(parents=True, exist_ok=True)

assert DIR_DATA.exists(), (
    f"Data directory '{DIR_DATA}' not found.\nRun 02_cepheids_extended_search.ipynb first (section 15)."
)
print(f"Data directory : {DIR_DATA.resolve()}")
print(f"LC directory   : {LC_DIR.resolve()}")
print(f"Figs directory : {DIR_FIGS.resolve()}")

# ── Analysis parameters (must match notebook 02) ──────────────────────────────
SNR_MIN = 3.0
BANDS = list("ugrizy")
PERIOD_MIN_DAYS = 0.3
PERIOD_MAX_DAYS = 200.0
LS_SAMPLES = 20
NSRC_MIN = 100

BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 12,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(DIR_FIGS / f"{name}.{ext}", bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Helper functions

In [ ]:
from astropy.timeseries import LombScargle, LombScargleMultiband


AB_FLUX_ZERO_NJY = 3631e9


def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert nJy flux -> AB magnitude and propagated error (vectorized)."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO_NJY), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def filter_lc(
    df_lc,
    mjd_col="r:midpointMjdTai",
    flux_col="r:scienceFlux",
    ferr_col="r:scienceFluxErr",
    band_col="r:band",
    snr_min=SNR_MIN,
):
    """
    Apply SNR cut and add mag/mag_err columns to a sources DataFrame.

    parameters:
    ===========
        df_lc : the light curve either src or fp
        mjd_col : column name with the relevant mjd name
        flux_col : column name with the relevant flux (default the scienceFlux for variable stars,
                   NOT psfFlux which is used for SNe difference-image photometry)
        ferr_col : column name with the relevant flux err
        band_col : band
        snr_min : cut on SNR
    """
    if df_lc.empty:
        return df_lc

    df = df_lc.copy()
    for col in (mjd_col, flux_col, ferr_col, band_col):
        if col not in df.columns:
            return pd.DataFrame()

    df[flux_col] = pd.to_numeric(df[flux_col], errors="coerce")
    df[ferr_col] = pd.to_numeric(df[ferr_col], errors="coerce")
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")

    snr = df[flux_col].abs() / df[ferr_col].replace(0, np.nan)

    df = df[snr >= snr_min].sort_values(mjd_col).reset_index(drop=True)
    df = df.dropna(subset=[flux_col, ferr_col, mjd_col]).reset_index(drop=True)

    mag, mag_err = flux_to_mag(df[flux_col].values, df[ferr_col].values)
    df["mag"] = mag
    df["mag_err"] = mag_err

    df = df[np.isfinite(df["mag"].values)].copy()
    if df.empty:
        return df
    return df.sort_values(mjd_col).reset_index(drop=True)


def lomb_scargle_period(mjd, mag, mag_err, pmin=PERIOD_MIN_DAYS, pmax=PERIOD_MAX_DAYS, samples=LS_SAMPLES):
    """Single-band Lomb-Scargle period search (magnitude space)."""
    mjd = np.asarray(mjd, dtype=float)
    mag = np.asarray(mag, dtype=float)
    mag_err = np.asarray(mag_err, dtype=float)
    mask = np.isfinite(mjd) & np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)
    t, y, dy = mjd[mask], mag[mask], mag_err[mask]
    if len(t) < 5:
        return np.nan, np.array([]), np.array([]), np.nan

    ls = LombScargle(t, y, dy)
    frequency, power = ls.autopower(
        minimum_frequency=1.0 / pmax,
        maximum_frequency=1.0 / pmin,
        samples_per_peak=samples,
    )
    best_freq = frequency[np.argmax(power)]
    best_period = 1.0 / best_freq
    try:
        fap = ls.false_alarm_probability(power.max(), method="baluev")
    except Exception:
        fap = np.nan
    return best_period, 1.0 / frequency, power, fap


def lomb_scargle_period_multiband(
    df_src: "pd.DataFrame",
    pmin: float = PERIOD_MIN_DAYS,
    pmax: float = PERIOD_MAX_DAYS,
    samples_per_peak: int = LS_SAMPLES,
):
    """
    Multiband Lomb-Scargle period search on a filtered sources DataFrame.

    Uses astropy.timeseries.LombScargleMultiband when more than one band is
    present, which fits a shared frequency with per-band amplitude/phase offsets.
    Falls back to single-band LombScargle when only one band is available.

    Parameters
    ----------
    df_src           : filtered sources DataFrame (output of filter_lc),
                       must contain columns r:midpointMjdTai, mag, mag_err,
                       r:band.
    pmin, pmax       : period search range in days.
    samples_per_peak : frequency grid oversampling factor.

    Returns
    -------
    best_period : float (days) or np.nan
    freq        : 1-D array of frequencies (1/day) or None
    power       : 1-D array of LS power or None
    fap         : false-alarm probability or np.nan (set later from per-band scan)
    n_bands     : int — number of bands actually used
    """
    if df_src.empty or len(df_src) < 5:
        return np.nan, None, None, np.nan, 0

    t = df_src["r:midpointMjdTai"].values
    mag = df_src["mag"].values
    mag_err = df_src["mag_err"].values
    bands = df_src["r:band"]

    mask = np.isfinite(t) & np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)
    t_arr, y_arr, dy_arr, b_arr = t[mask], mag[mask], mag_err[mask], bands[mask]
    if len(t_arr) < 5:
        return np.nan, None, None, np.nan, 0

    fmin = 1.0 / pmax
    fmax = 1.0 / pmin

    # Keep only bands with enough points to constrain a sinusoid.
    bands_ok = [b for b in BANDS if np.sum(b_arr == b) >= 3]
    n_bands = len(bands_ok)
    use_band = np.isin(b_arr, bands_ok)

    # Multiband path: one common frequency, with per-band offsets/amplitudes.
    if n_bands >= 2 and np.sum(use_band) >= 5:
        try:
            lsm = LombScargleMultiband(
                t_arr[use_band],
                y_arr[use_band],
                b_arr[use_band],
                dy_arr[use_band],
                nterms_base=1,
                nterms_band=1,
            )
            freq, power = lsm.autopower(
                minimum_frequency=fmin,
                maximum_frequency=fmax,
                samples_per_peak=samples_per_peak,
            )
            best_freq = freq[np.argmax(power)]
            best_period = 1.0 / best_freq if best_freq > 0 else np.nan

            # Astropy does not provide a direct FAP for LombScargleMultiband.
            # The retained FAP is selected later from independent per-band scans
            # (see lomb_scargle_period_optimizeinallbands).
            fap = np.nan

            return best_period, freq, power, fap, n_bands
        except Exception as exc:
            print(f"    [LS multiband fallback] {exc}")

    # Single-band fallback: prefer r-band; use all bands only if r is too sparse.
    if "r:band" in df_src.columns:
        df_r = df_src[df_src["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_src
    else:
        df_r = df_src

    if len(df_r) < 5:
        return np.nan, None, None, np.nan, 1

    best_period, period, power, fap = lomb_scargle_period(
        df_r["r:midpointMjdTai"].values,
        df_r["mag"].values,
        df_r["mag_err"].values,
        pmin=pmin,
        pmax=pmax,
        samples=samples_per_peak,
    )
    freq = 1.0 / period if len(period) else np.array([])

    return best_period, freq, power, fap, 1


def lomb_scargle_period_optimizeinallbands(
    df_src: "pd.DataFrame",
    pmin: float = PERIOD_MIN_DAYS,
    pmax: float = PERIOD_MAX_DAYS,
    samples_per_peak: int = LS_SAMPLES,
):
    """
    Run independent single-band Lomb-Scargle searches for each usable band.

    Parameters
    ----------
    df_src           : filtered sources DataFrame (output of filter_lc),
                       must contain columns r:midpointMjdTai, mag, mag_err,
                       r:band.
    pmin, pmax       : period search range in days.
    samples_per_peak : frequency grid oversampling factor.

    Returns
    -------
    pd.DataFrame with one row per band and columns band_ok, band_period,
    band_fap, and band_npoints.
    """
    if df_src.empty or len(df_src) < 5 or "r:band" not in df_src.columns:
        return pd.DataFrame(columns=["band_ok", "band_period", "band_fap", "band_npoints"])

    band_select = []
    band_period = []
    band_fap = []
    band_npoints = []

    for band in BANDS:
        df_b = df_src[df_src["r:band"] == band]
        if len(df_b) < 5:
            continue

        best_period_b, _, _, fap_b = lomb_scargle_period(
            df_b["r:midpointMjdTai"].values,
            df_b["mag"].values,
            df_b["mag_err"].values,
            pmin=pmin,
            pmax=pmax,
            samples=samples_per_peak,
        )

        band_select.append(band)
        band_period.append(best_period_b)
        band_fap.append(fap_b)
        band_npoints.append(len(df_b))

    results = {
        "band_ok": band_select,
        "band_period": band_period,
        "band_fap": band_fap,
        "band_npoints": band_npoints,
    }

    return pd.DataFrame(results)


def phase_fold(mjd, period, t0=None):
    """Return phase in [0, 1)."""
    if t0 is None:
        t0 = np.nanmin(mjd)
    return ((mjd - t0) / period) % 1.0


print("Helper functions defined.")
print(
    "  filter_lc()                          — SNR cut + mag conversion, flux_col='r:scienceFlux' (NOT psfFlux)"
)
print("  lomb_scargle_period()                 — single-band LS (magnitude space)")
print("  lomb_scargle_period_multiband()       — LombScargleMultiband, fallback to single-band")
print("  lomb_scargle_period_optimizeinallbands() — independent per-band LS scan, used for robust FAP")

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.

    Uses astropy.time.Time for the conversion.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40 degrees to the left for readability.
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=12, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

## 3. Reload summary DataFrames

In [ ]:
def _load_parquet(path: pathlib.Path, label: str) -> pd.DataFrame:
    if path.exists():
        df = pd.read_parquet(path)
        print(f"Loaded {label:25s}: {len(df):,} rows  |  cols: {list(df.columns)[:6]}...")
        return df
    print(f"NOT FOUND: {path}  ({label})")
    return pd.DataFrame()


df_obj = _load_parquet(DIR_DATA / "df_obj.parquet", "df_obj")
df_pulsators = _load_parquet(DIR_DATA / "df_pulsators.parquet", "df_pulsators")
df_periods = _load_parquet(DIR_DATA / "df_periods.parquet", "df_periods")
df_meta = _load_parquet(DIR_DATA / "lc_dict_meta.parquet", "lc_dict_meta")

print(f"\nTotal objects in survey    : {len(df_obj):,}")
print(f"Pulsating variable candidates: {len(df_pulsators):,}")
print(f"Objects with saved LC      : {len(df_meta):,}")

## 4. Reload individual light curves into `lc_dict`

In [ ]:
lc_dict = {}

if df_meta.empty:
    print("No metadata found — lc_dict will be empty.")
else:
    for _, row in df_meta.iterrows():
        oid = row["diaObjectId"]
        obj_dir = LC_DIR / str(oid)

        # Sources
        src_path = obj_dir / "sources.parquet"
        df_src = pd.read_parquet(src_path) if src_path.exists() else pd.DataFrame()
        if len(df_src) < NSRC_MIN:
            print(f"{oid} :: {len(df_src)}")
            continue

        mag = df_src["r:scienceFlux"].values
        mag = mag[np.isfinite(mag)]
        if len(mag) == 0:
            print(f"{oid} :: only nan")
            continue

        # Forced photometry
        fp_path = obj_dir / "fp.parquet"
        df_fp = pd.read_parquet(fp_path) if fp_path.exists() else pd.DataFrame()

        # Metadata (all scalar columns except diaObjectId, n_src, n_fp)
        meta = row.drop(labels=["diaObjectId", "n_src", "n_fp"], errors="ignore").to_dict()

        lc_dict[oid] = {"src": df_src, "fp": df_fp, "meta": meta}

    print(f"Reloaded {len(lc_dict)} light curves from disk.")
    for oid, data in list(lc_dict.items())[:5]:
        print(
            f"  {oid}  src={len(data['src'])}  fp={len(data['fp'])}  "
            f"class={data['meta'].get('pulsator_class', '?')}"
        )

## 5. Summary statistics

In [ ]:
if not df_pulsators.empty:
    print("=" * 60)
    print("PULSATING VARIABLE CANDIDATES")
    print("=" * 60)
    if "pulsator_class" in df_pulsators.columns:
        print(df_pulsators["pulsator_class"].value_counts().to_string())
    if "field" in df_pulsators.columns:
        print("\nBy field:")
        print(df_pulsators["field"].value_counts().to_string())
    display(df_pulsators.head(10))
else:
    print("df_pulsators is empty.")

## 6. Raw light curves — overview plot

In [ ]:
if not lc_dict:
    print("No light curves available.")
else:
    NC_PLOT = min(50, len(lc_dict))
    ncols = 3
    nrows = int(np.ceil(NC_PLOT / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (oid, data) in enumerate(list(lc_dict.items())[:NC_PLOT]):
        ax = axes[idx]
        meta = data["meta"]
        df_filt = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()

        if df_filt.empty:
            ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        else:
            mag = df_filt["mag"].values
            mag = mag[np.isfinite(mag)]
            mag_min = np.percentile(mag, 5) - 0.3
            mag_max = np.percentile(mag, 95) + 0.3

            for band in BANDS:
                dfb = df_filt[df_filt["r:band"] == band]
                if dfb.empty:
                    continue
                ax.errorbar(
                    dfb["r:midpointMjdTai"],
                    dfb["mag"],
                    dfb["mag_err"],
                    fmt="o",
                    ms=4,
                    lw=0.5,
                    color=BAND_COLORS.get(band, "grey"),
                    label=band,
                    alpha=0.8,
                )
            # ax.invert_yaxis()
            ax.set_ylim(mag_max, mag_min)
            add_date_axis_on_top(ax, df_filt["r:midpointMjdTai"])

        pclass = meta.get("pulsator_class", "?")
        stype = meta.get("f:xm_simbad_otype", "?")
        vsx = meta.get("f:xm_vsx_Type", "?")
        ax.set_title(f"{oid}\n{pclass} | SIMBAD:{stype} VSX:{vsx}", fontsize=7)
        ax.set_xlabel("MJD", fontsize=7)
        ax.set_ylabel("AB mag", fontsize=7)
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[NC_PLOT:]:
        ax.set_visible(False)

    plt.suptitle("Raw light curves — selected pulsators (reloaded)", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_raw_lc")
    plt.show()

## 7. Source Photometry and Forced-photometry light curves

In [ ]:
objects_with_fp = {oid: data for oid, data in lc_dict.items() if not data["fp"].empty}
print(f"{len(objects_with_fp)} objects have forced-photometry data.")

if objects_with_fp:
    NC_PLOT = min(100, len(objects_with_fp))
    ncols = 4
    nrows = int(np.ceil(NC_PLOT / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (oid, data) in enumerate(list(objects_with_fp.items())[:NC_PLOT]):
        ax = axes[idx]
        meta = data["meta"]

        df_src = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()
        df_fp = data["fp"].copy()

        mag = df_src["mag"].values
        mag = mag[np.isfinite(mag)]

        if len(mag) == 0:  # ← guard ajouté
            ax.text(
                0.5,
                0.5,
                "No finite mag",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=8,
                color="red",
            )
        else:
            mag_min = np.percentile(mag, 5) - 0.3
            mag_max = np.percentile(mag, 95) + 0.3

            for band in BANDS:
                dfb = df_src[df_src["r:band"] == band]
                if dfb.empty:
                    continue
                ax.errorbar(
                    dfb["r:midpointMjdTai"],
                    dfb["mag"],
                    dfb["mag_err"],
                    fmt="o",
                    ms=4,
                    lw=0.5,
                    color=BAND_COLORS.get(band, "grey"),
                    label=band,
                    alpha=0.8,
                )
            # ax.invert_yaxis()
            ax.set_ylim(mag_max, mag_min)
            add_date_axis_on_top(ax, df_src["r:midpointMjdTai"])

        # Determine flux column available in FP data.
        # Use scienceFlux for variable stars (NOT psfFlux, which is for SNe).
        flux_col = None
        for col in ("r:scienceFlux", "forcedSourceFlux", "fp_flux"):
            if col in df_fp.columns:
                flux_col = col
                break

        if flux_col is None or "r:midpointMjdTai" not in df_fp.columns:
            ax.text(
                0.5, 0.5, "No usable FP columns", ha="center", va="center", transform=ax.transAxes, fontsize=7
            )
        else:
            err_col = flux_col.replace("Flux", "FluxErr").replace("flux", "fluxErr")
            if err_col not in df_fp.columns:
                err_col = flux_col  # fallback
            band_col = "r:band" if "r:band" in df_fp.columns else None

            if band_col:
                for band in BANDS:
                    dfb = df_fp[df_fp[band_col] == band]
                    if dfb.empty:
                        continue
                    mag, mag_err = flux_to_mag(dfb[flux_col].values, dfb[err_col].values)
                    mask = np.isfinite(mag)
                    ax.errorbar(
                        dfb["r:midpointMjdTai"].values[mask],
                        mag[mask],
                        mag_err[mask],
                        fmt="s",
                        ms=2,
                        lw=0.5,
                        # markerfacecolor="white",
                        markerfacecolor="none",
                        markeredgecolor=BAND_COLORS.get(band, "grey"),
                        markeredgewidth=1.3,
                        # color=BAND_COLORS.get(band, "grey"),
                        label=band,
                        alpha=0.7,
                    )
            else:
                mag, mag_err = flux_to_mag(df_fp[flux_col].values, df_fp[err_col].values)
                mask = np.isfinite(mag)
                ax.errorbar(
                    df_fp["r:midpointMjdTai"].values[mask],
                    mag[mask],
                    mag_err[mask],
                    fmt="s",
                    ms=2,
                    # color="grey",
                    markerfacecolor="white",
                    markeredgecolor="grey",
                    markeredgewidth=1.3,
                    alpha=0.7,
                )
            # ax.invert_yaxis()

        pclass = meta.get("pulsator_class", "?")
        ax.set_title(f"{oid}\n{pclass}", fontsize=7)
        ax.set_xlabel("MJD", fontsize=7)
        ax.set_ylabel("AB mag", fontsize=7)
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[NC_PLOT:]:
        ax.set_visible(False)

    plt.suptitle("Forced-photometry light curves (reloaded)", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_src_fp_lc")
    plt.show()

## 8. Period search (recompute or reload)

If `df_periods.parquet` was saved by notebook 02, it is used directly.
Otherwise periods are recomputed from the reloaded sources.

In [ ]:
if df_periods.empty:
    print("df_periods not found — recomputing periods with multiband Lomb-Scargle ...")
    period_results = []

    for oid, data in lc_dict.items():
        if data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if len(df_filt) < 100:
            continue
        meta = data["meta"]

        # Prefer the period found by LombScargleMultiband.
        best_period, _, _, _, n_bands = lomb_scargle_period_multiband(df_filt)

        # Robust FAP: independent per-band LS scan, take the minimum valid FAP
        # (mirrors notebook 02 — LombScargleMultiband has no native FAP).
        dflombsc_results = lomb_scargle_period_optimizeinallbands(df_filt)
        valid_fap = pd.to_numeric(dflombsc_results.get("band_fap", pd.Series(dtype=float)), errors="coerce")
        valid_fap = valid_fap[np.isfinite(valid_fap) & (valid_fap > 0)]
        fap = valid_fap.min() if len(valid_fap) else np.nan

        period_results.append(
            {
                "diaObjectId": oid,
                "field": meta.get("field", ""),
                "pulsator_class": meta.get("pulsator_class", ""),
                "simbad_otype": meta.get("f:xm_simbad_otype", ""),
                "vsx_type": meta.get("f:xm_vsx_Type", ""),
                "gcvs_type": meta.get("f:xm_gcvs_type", ""),
                "best_period_d": best_period,
                "ls_fap": fap,
                "n_pts": len(df_filt),
                "n_bands": n_bands,
            }
        )
        print(
            f"  {oid}  P={best_period:.3f}d  FAP={fap:.1e}  "
            f"bands={n_bands}  ({meta.get('pulsator_class', '?')})"
        )

    df_periods = pd.DataFrame(period_results)
    if not df_periods.empty:
        df_periods.to_parquet(DIR_DATA / "df_periods.parquet", index=False)
        print(f"Saved recomputed df_periods ({len(df_periods)} rows).")
else:
    print(f"Using precomputed df_periods ({len(df_periods)} rows).")

if not df_periods.empty:
    display(df_periods.sort_values("best_period_d"))

## 9. Phase-folded light curves

In [ ]:
if df_periods.empty:
    print("No period data available.")
else:
    df_plot = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(12)
    ncols = 3
    nrows = int(np.ceil(len(df_plot) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_plot.iterrows()):
        oid = row["diaObjectId"]
        period = row["best_period_d"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if df_filt.empty:
            continue

        ax = axes[idx]
        t0 = df_filt["r:midpointMjdTai"].min()
        for band in BANDS:
            dfb = df_filt[df_filt["r:band"] == band]
            if dfb.empty:
                continue
            phi = phase_fold(dfb["r:midpointMjdTai"].values, period, t0)
            ax.errorbar(
                np.concatenate([phi, phi + 1]),
                np.concatenate([dfb["mag"].values] * 2),
                yerr=np.concatenate([dfb["mag_err"].values] * 2),
                fmt="o",
                ms=3,
                lw=0.5,
                color=BAND_COLORS.get(band, "grey"),
                label=band,
                alpha=0.85,
            )
        ax.set_xlim(0, 2)
        ax.invert_yaxis()
        ax.set_xlabel("Phase")
        ax.set_ylabel("AB mag")
        ax.set_title(
            f"{oid}  P={period:.3f}d  FAP={row['ls_fap']:.1e}\n"
            f"{row['pulsator_class']} | {row.get('vsx_type', '?')}",
            fontsize=7,
        )
        ax.legend(fontsize=6, ncol=3)

    for ax in axes[len(df_plot) :]:
        ax.set_visible(False)

    plt.suptitle("Phase-folded light curves (sorted by LS FAP) — reloaded", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_phased_lc")
    plt.show()

## 10. Lomb-Scargle power spectrum for individual objects

In [ ]:
# Show LS periodogram for the top-N most significant objects
if not df_periods.empty:
    TOP_N = 20
    df_top = df_periods.dropna(subset=["best_period_d"]).sort_values("ls_fap").head(TOP_N)
    ncols = 2
    nrows = int(np.ceil(TOP_N / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for idx, (_, row) in enumerate(df_top.iterrows()):
        oid = row["diaObjectId"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if len(df_filt) < 5:
            continue

        df_r = df_filt[df_filt["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_filt

        # Note: lomb_scargle_period() returns the period grid directly
        # (1/frequency) as its 3rd element, not the frequency grid.
        _, period_grid, power, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
        if period_grid is None or len(period_grid) == 0:
            continue

        ax = axes[idx]
        ax.semilogx(period_grid, power, lw=0.8, color="steelblue")
        ax.axvline(row["best_period_d"], color="red", lw=1.5, ls="--", label=f"P={row['best_period_d']:.3f}d")
        ax.set_xlabel("Period (days)")
        ax.set_ylabel("LS power")
        ax.set_title(f"{oid}  FAP={row['ls_fap']:.1e}\n{row['pulsator_class']}", fontsize=8)
        ax.legend(fontsize=7)

    for ax in axes[TOP_N:]:
        ax.set_visible(False)

    plt.suptitle("Lomb-Scargle periodograms — top objects", y=1.01, fontsize=10)
    plt.tight_layout()
    savefig("pulsators_ls_periodogram")
    plt.show()
else:
    print("No period data available.")

## 11. Period–Luminosity diagram

In [ ]:
if df_periods.empty or df_periods["best_period_d"].isna().all():
    print("No period data available.")
else:
    # Add median r-band magnitude from reloaded sources
    mag_med = []
    for oid in df_periods["diaObjectId"]:
        data = lc_dict.get(oid, {})
        if data and not data["src"].empty:
            df_r = filter_lc(data["src"])
            df_r = df_r[df_r["r:band"] == "r"]
            mag_med.append(np.nanmedian(df_r["mag"]) if len(df_r) > 0 else np.nan)
        else:
            mag_med.append(np.nan)

    df_periods = df_periods.copy()
    df_periods["mag_r_median"] = mag_med

    df_pl = df_periods.dropna(subset=["best_period_d", "mag_r_median"])
    df_pl = df_pl[(df_pl["best_period_d"] > 0.1) & (df_pl["best_period_d"] < 200)]

    class_marker = {
        "cepheid_classical": ("*", "red", 100),
        "cepheid_type2": ("^", "darkorange", 80),
        "rr_lyrae": ("o", "dodgerblue", 40),
        "delta_scuti": ("s", "green", 30),
        "lpv_mira": ("D", "purple", 40),
        "rv_tauri": ("P", "brown", 50),
        "other_pulsator": ("x", "grey", 25),
    }

    fig, ax = plt.subplots(figsize=(9, 6))
    for cls, (marker, color, size) in class_marker.items():
        sub = df_pl[df_pl["pulsator_class"] == cls]
        if sub.empty:
            continue
        ax.scatter(
            np.log10(sub["best_period_d"]),
            sub["mag_r_median"],
            marker=marker,
            color=color,
            s=size,
            alpha=0.8,
            label=f"{cls} (N={len(sub)})",
            zorder=3,
        )

    # Leavitt law reference (rough)
    logP = np.linspace(0, 2, 50)
    ax.plot(logP, -2.81 * logP + 11.5, "k--", lw=1, alpha=0.4, label="Leavitt law (rough, DM≈11)")

    ax.set_xlabel("log₁₀(Period / days)")
    ax.set_ylabel("Median r-band AB mag (apparent)")
    ax.invert_yaxis()
    ax.set_title(
        "Period–Luminosity diagram — Pulsating variables in LSST fields\n"
        "(apparent magnitudes, no distance or extinction correction)"
    )
    ax.legend(fontsize=8, loc="best")
    plt.tight_layout()
    savefig("pulsators_PL_diagram")
    plt.show()

In [ ]:
# Dans une cellule de diagnostic :
first_oid = next(iter(lc_dict))
df_fp_test = lc_dict[first_oid]["fp"]
print(df_fp_test.columns.tolist())
print(df_fp_test.head(3))

## 12. Interactive single-object explorer

Three-panel view for the best-detected object (lowest LS FAP).
**Marker convention** — same band colour throughout:

| Symbol | Meaning |
|--------|---------|
| filled circle `●` | DIA detection (sources) |
| open circle `○` (white face, coloured edge) | Forced photometry |

- **Topt** : light curve (src + FP on the same axes, vs MJD)
- **Bottom-left** : Lomb-Scargle periodogram
- **Bottom-right** : phase-folded light curve (src + FP on the same axes, × 2 periods)

Set `EXPLORE_OID` to any `diaObjectId` present in `lc_dict` to inspect a different object.


In [ ]:
# ── Section 12: single-object explorer ───────────────────────────────────────


def _fp_mag_by_band(df_fp):
    """
    Extract per-band (mjd, mag, mag_err) arrays from a forced-photometry DataFrame.
    Returns dict {band: (mjd_arr, mag_arr, mag_err_arr)}, empty if data absent.
    """
    result = {}
    if df_fp.empty or "r:midpointMjdTai" not in df_fp.columns:
        return result
    # Use scienceFlux for variable stars (NOT psfFlux, which is for SNe).
    flux_col = next(
        (c for c in ("r:scienceFlux", "forcedSourceFlux", "fp_flux") if c in df_fp.columns),
        None,
    )
    if flux_col is None:
        return result
    err_col = flux_col.replace("Flux", "FluxErr").replace("flux", "fluxErr")
    if err_col not in df_fp.columns:
        err_col = flux_col
    band_col = "r:band" if "r:band" in df_fp.columns else None
    rows = [df_fp] if band_col is None else [(band, df_fp[df_fp[band_col] == band]) for band in BANDS]
    for item in rows:
        if band_col is None:
            band, dfb = "?", item
        else:
            band, dfb = item
        if dfb.empty:
            continue
        mag, mag_err = flux_to_mag(dfb[flux_col].values, dfb[err_col].values)
        mask = np.isfinite(mag)
        if mask.any():
            result[band] = (
                dfb["r:midpointMjdTai"].values[mask],
                mag[mask],
                mag_err[mask],
            )
    return result


if not lc_dict:
    print("lc_dict is empty — nothing to explore.")
else:
    # ── Object selection ──────────────────────────────────────────────────────
    # Default: best LS detection.  Override: EXPLORE_OID = <some diaObjectId>
    if not df_periods.empty and "ls_fap" in df_periods.columns:
        EXPLORE_OID = df_periods.dropna(subset=["ls_fap"]).sort_values("ls_fap")["diaObjectId"].iloc[0]
    else:
        EXPLORE_OID = next(iter(lc_dict))

    print(f"Exploring object: {EXPLORE_OID}")
    data = lc_dict[EXPLORE_OID]
    meta = data["meta"]
    df_src = filter_lc(data["src"]) if not data["src"].empty else pd.DataFrame()
    fp_by_band = _fp_mag_by_band(data["fp"])

    n_src = len(df_src)
    n_fp = sum(len(v[0]) for v in fp_by_band.values())
    print(f"  src detections : {n_src}")
    print(f"  FP epochs      : {n_fp}")

    # ── Figure layout: 2×2 grid, bottom row merged ────────────────────────────
    fig = plt.figure(figsize=(12, 8))
    gs = gridspec.GridSpec(2, 2)
    # Top row (merged)
    ax_lc = fig.add_subplot(gs[0, :])  # top  : light curve
    # Bottom row
    ax_ls = fig.add_subplot(gs[1, 0])  # bottom-left : LS periodogram
    ax_ph = fig.add_subplot(gs[1, 1])  # bottom-right : phase (full width)

    mag_min = 16
    mag_max = 27
    # ── Panel 1 — Light curve: src (filled ●) + FP (open ○) ─────────────────
    if not df_src.empty:
        mag = df_src["mag"].values
        mag = mag[np.isfinite(mag)]
        if len(mag) == 0:
            ax_lc.text(
                0.5,
                0.5,
                "No finite mag",
                ha="center",
                va="center",
                transform=ax_lc.transAxes,
                fontsize=8,
                color="red",
            )
        else:
            mag_min = np.percentile(mag, 5) - 0.3
            mag_max = np.percentile(mag, 95) + 0.3

            for band in BANDS:
                dfb = df_src[df_src["r:band"] == band]
                if dfb.empty:
                    continue
                col = BAND_COLORS.get(band, "grey")
                ax_lc.errorbar(
                    dfb["r:midpointMjdTai"],
                    dfb["mag"],
                    dfb["mag_err"],
                    fmt="o",
                    ms=4,
                    lw=0.5,
                    color=col,
                    markerfacecolor=col,
                    markeredgecolor=col,
                    alpha=0.85,
                    label=f"{band} src",
                )

    for band, (mjd, mag, mag_err) in fp_by_band.items():
        col = BAND_COLORS.get(band, "grey")
        ax_lc.errorbar(
            mjd,
            mag,
            mag_err,
            fmt="v",
            ms=5,
            lw=0.5,
            color=col,
            markerfacecolor="white",
            markeredgecolor=col,
            markeredgewidth=1.3,
            alpha=0.80,
            label=f"{band} FP",
        )

    if not df_src.empty or fp_by_band:
        pass
        # ax_lc.invert_yaxis()

    ax_lc.set_xlabel("MJD")
    ax_lc.set_ylabel("AB mag")
    ax_lc.set_title(f"Light curve — {EXPLORE_OID}")
    ax_lc.legend(
        fontsize=10,
        ncol=6,
        title="● src   ○ forced phot.",
        title_fontsize=10,
        loc="best",
        framealpha=0.7,
    )
    ax_lc.set_ylim(mag_max, mag_min)
    add_date_axis_on_top(ax_lc, df_src["r:midpointMjdTai"])

    # ── Panel 2 — Lomb-Scargle periodogram ───────────────────────────────────
    if not df_src.empty and len(df_src) >= 5:
        df_r = df_src[df_src["r:band"] == "r"]
        if len(df_r) < 5:
            df_r = df_src
        # Note: lomb_scargle_period() returns the period grid directly
        # (1/frequency) as its 3rd element, not the frequency grid.
        best_period, period_grid, power, fap = lomb_scargle_period(
            df_r["r:midpointMjdTai"].values,
            df_r["mag"].values,
            df_r["mag_err"].values,
        )
        if period_grid is not None and len(period_grid) > 0:
            ax_ls.semilogx(period_grid, power, lw=0.8, color="steelblue")
            if np.isfinite(best_period):
                ax_ls.axvline(
                    best_period,
                    color="red",
                    lw=1.5,
                    ls="--",
                    label=f"P = {best_period:.4f} d\nFAP = {fap:.2e}",
                )
            ax_ls.legend(fontsize=8)
    ax_ls.set_xlabel("Period (days)")
    ax_ls.set_ylabel("LS power")
    ax_ls.set_title("Lomb-Scargle periodogram (r-band or all)")

    # ── Panel 3 — Phase-folded: src (filled ●) + FP (open ○) ────────────────
    row_p = df_periods[df_periods["diaObjectId"] == EXPLORE_OID] if not df_periods.empty else pd.DataFrame()
    if not row_p.empty and np.isfinite(row_p.iloc[0]["best_period_d"]):
        P = row_p.iloc[0]["best_period_d"]
        t0 = df_src["r:midpointMjdTai"].min() if not df_src.empty else 0.0

        # Sources — filled circles
        if not df_src.empty:
            for band in BANDS:
                dfb = df_src[df_src["r:band"] == band]
                if dfb.empty:
                    continue
                col = BAND_COLORS.get(band, "grey")
                phi = phase_fold(dfb["r:midpointMjdTai"].values, P, t0)
                phi2 = np.concatenate([phi, phi + 1])
                mag2 = np.concatenate([dfb["mag"].values] * 2)
                err2 = np.concatenate([dfb["mag_err"].values] * 2)
                ax_ph.errorbar(
                    phi2,
                    mag2,
                    err2,
                    fmt="o",
                    ms=4,
                    lw=0.5,
                    color=col,
                    markerfacecolor=col,
                    markeredgecolor=col,
                    alpha=0.85,
                    label=f"{band} src",
                )

        # Forced photometry — open circles
        for band, (mjd, mag, mag_err) in fp_by_band.items():
            col = BAND_COLORS.get(band, "grey")
            phi = phase_fold(mjd, P, t0)
            phi2 = np.concatenate([phi, phi + 1])
            mag2 = np.concatenate([mag, mag])
            err2 = np.concatenate([mag_err, mag_err])
            ax_ph.errorbar(
                phi2,
                mag2,
                err2,
                fmt="o",
                ms=5,
                lw=0.5,
                color=col,
                markerfacecolor="white",
                markeredgecolor=col,
                markeredgewidth=1.3,
                alpha=0.80,
                label=f"{band} FP",
            )

        ax_ph.set_xlim(0, 2)
        ax_ph.invert_yaxis()
        ax_ph.set_title(
            f"Phase-folded light curve  —  P = {P:.4f} d  (× 2 periods)",
            fontsize=11,
        )
        ax_ph.legend(
            fontsize=7,
            ncol=6,
            title="● src   ○ forced phot.",
            title_fontsize=7,
            loc="best",
            framealpha=0.7,
        )
    else:
        ax_ph.text(
            0.5,
            0.5,
            "No period available",
            ha="center",
            va="center",
            transform=ax_ph.transAxes,
            fontsize=12,
        )
    ax_ph.set_xlabel("Phase", fontsize=11)
    ax_ph.set_ylabel("AB mag", fontsize=11)
    ax_ph.set_ylim(mag_max, mag_min)

    # ── Suptitle ──────────────────────────────────────────────────────────────
    pclass = meta.get("pulsator_class", "?")
    stype = meta.get("f:xm_simbad_otype", "?")
    vsx = meta.get("f:xm_vsx_Type", "?")
    plt.suptitle(
        f"{EXPLORE_OID}  —  {pclass}  |  SIMBAD: {stype}  |  VSX: {vsx}\n"
        f"src detections: {n_src}   FP epochs: {n_fp}",
        fontsize=11,
        y=1.01,
    )
    plt.tight_layout()
    savefig(f"explorer_{EXPLORE_OID}")
    plt.show()

## 13. Period histograms by pulsator class

Do the different variable-star families separate cleanly in period space?  
This matters for photometric-calibration work: we want families with
**well-defined, narrow period ranges** so that a phase-folded flux model
can be constructed and compared to new observations.

Known period ranges (literature):

| Class | Typical period range |
|---|---|
| Delta Scuti | 0.02 – 0.3 d |
| RR Lyrae (ab) | 0.3 – 1.0 d |
| RR Lyrae (c)  | 0.2 – 0.5 d |
| W Vir / Type II Cep | 1 – 35 d |
| Classical Cepheids (DCEP) | 1 – 100 d |
| Mira / LPV | 100 – 1000 d |


In [ ]:
# ── Section 13: period histograms ────────────────────────────────────────────
import matplotlib.patches as mpatches

# Reference period ranges (days) for vertical shading
PERIOD_BANDS_REF = [
    (0.02, 0.30, "#f0e68c", "Delta Scuti"),
    (0.30, 1.00, "#add8e6", "RR Lyrae"),
    (1.00, 35.0, "#90ee90", "W Vir / Type II Cep"),
    (1.00, 100.0, "#ffa07a", "Classical Cepheids"),
    (100.0, 500.0, "#d8bfd8", "Mira / LPV"),
]

CLASS_COLORS = {
    "cepheid_classical": "red",
    "cepheid_type2": "darkorange",
    "rr_lyrae": "dodgerblue",
    "delta_scuti": "green",
    "lpv_mira": "purple",
    "rv_tauri": "brown",
    "other_pulsator": "grey",
}

if df_periods.empty or "best_period_d" not in df_periods.columns:
    print("No period data — skipping histograms.")
else:
    df_p = df_periods.dropna(subset=["best_period_d"]).copy()
    df_p = df_p[df_p["best_period_d"] > 0]
    classes_present = [c for c in CLASS_COLORS if c in df_p["pulsator_class"].values]

    # ── Figure 1: stacked histogram on log-period axis ────────────────────────
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 1.5]})

    ax_hist = axes[0]
    ax_strip = axes[1]

    bins = np.logspace(
        np.log10(max(0.01, df_p["best_period_d"].min() * 0.8)),
        np.log10(min(600.0, df_p["best_period_d"].max() * 1.2)),
        40,
    )

    # Reference background bands
    for plo, phi, col, label in PERIOD_BANDS_REF:
        ax_hist.axvspan(plo, phi, alpha=0.10, color=col, zorder=0)
        ax_strip.axvspan(plo, phi, alpha=0.10, color=col, zorder=0)

    # Stacked histogram
    data_stacked = [df_p[df_p["pulsator_class"] == c]["best_period_d"].values for c in classes_present]
    colors_stacked = [CLASS_COLORS[c] for c in classes_present]

    ax_hist.hist(
        data_stacked,
        bins=bins,
        stacked=True,
        color=colors_stacked,
        label=classes_present,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.4,
    )
    ax_hist.set_xscale("log")
    ax_hist.set_xlabel("Period (days)", fontsize=11)
    ax_hist.set_ylabel("Number of objects", fontsize=11)
    ax_hist.set_title("Period distribution by pulsator class", fontsize=12)
    ax_hist.legend(fontsize=9, loc="upper right")

    # Secondary x-axis: log10 P
    ax2 = ax_hist.twiny()
    ax2.set_xscale("log")
    ax2.set_xlim(ax_hist.get_xlim())
    ax2.set_xlabel("log₁₀(Period / days)", fontsize=9)
    logP_ticks = [0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]
    ax2.set_xticks([t for t in logP_ticks if ax_hist.get_xlim()[0] <= t <= ax_hist.get_xlim()[1]])
    ax2.set_xticklabels(
        [f"{np.log10(t):.1f}" for t in logP_ticks if ax_hist.get_xlim()[0] <= t <= ax_hist.get_xlim()[1]],
        fontsize=8,
    )

    # ── Strip plot: individual periods per class (jittered) ──────────────────
    for k, cls in enumerate(classes_present):
        vals = df_p[df_p["pulsator_class"] == cls]["best_period_d"].values
        if len(vals) == 0:
            continue
        jitter = np.random.default_rng(42).uniform(-0.3, 0.3, len(vals))
        ax_strip.scatter(
            vals,
            np.full(len(vals), k) + jitter,
            color=CLASS_COLORS[cls],
            s=20,
            alpha=0.7,
            zorder=3,
        )
    ax_strip.set_xscale("log")
    ax_strip.set_yticks(range(len(classes_present)))
    ax_strip.set_yticklabels(classes_present, fontsize=9)
    ax_strip.set_xlabel("Period (days)", fontsize=11)
    ax_strip.set_title("Individual periods (strip plot)", fontsize=10)
    ax_strip.set_xlim(ax_hist.get_xlim())

    plt.tight_layout()
    savefig("period_histogram_by_class")
    plt.show()

    # ── Figure 2: per-class KDE on log P ─────────────────────────────────────
    try:
        from scipy.stats import gaussian_kde

        HAS_SCIPY = True
    except ImportError:
        HAS_SCIPY = False
        print("scipy not available — KDE plot skipped")

    if HAS_SCIPY:
        log_bins = np.linspace(
            np.log10(max(0.01, df_p["best_period_d"].min() * 0.8)),
            np.log10(min(600.0, df_p["best_period_d"].max() * 1.2)),
            300,
        )
        fig, ax = plt.subplots(figsize=(10, 4))
        for plo, phi, col, label in PERIOD_BANDS_REF:
            ax.axvspan(np.log10(plo), np.log10(phi), alpha=0.09, color=col, zorder=0)
        for cls in classes_present:
            vals = np.log10(df_p[df_p["pulsator_class"] == cls]["best_period_d"].values)
            if len(vals) < 3:
                continue
            kde = gaussian_kde(vals, bw_method="scott")
            ax.plot(log_bins, kde(log_bins), color=CLASS_COLORS[cls], lw=2, label=f"{cls} (N={len(vals)})")
            ax.fill_between(log_bins, kde(log_bins), alpha=0.15, color=CLASS_COLORS[cls])
        ax.set_xlabel("log₁₀(Period / days)", fontsize=11)
        ax.set_ylabel("KDE density", fontsize=11)
        ax.set_title("KDE of period distribution by pulsator class", fontsize=12)
        ax.legend(fontsize=9)
        # Annotate reference bands
        for plo, phi, _, label in PERIOD_BANDS_REF:
            xmid = (np.log10(plo) + np.log10(phi)) / 2
            xlim = ax.get_xlim()
            if xlim[0] < xmid < xlim[1]:
                ax.text(
                    xmid,
                    ax.get_ylim()[1] * 0.95,
                    label,
                    ha="center",
                    va="top",
                    fontsize=7,
                    color="#555",
                    style="italic",
                )
        plt.tight_layout()
        savefig("period_kde_by_class")
        plt.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    print("\nPeriod statistics by pulsator class:")
    print("=" * 72)
    summary_rows = []
    for cls in CLASS_COLORS:
        sub = df_p[df_p["pulsator_class"] == cls]["best_period_d"]
        if len(sub) == 0:
            continue
        summary_rows.append(
            {
                "class": cls,
                "N": len(sub),
                "P_min_d": sub.min(),
                "P_median_d": sub.median(),
                "P_max_d": sub.max(),
                "P_std_d": sub.std(),
            }
        )
    df_summary = pd.DataFrame(summary_rows)
    display(df_summary.round(3))

## 14. Selection and Fourier fitting of photometric-calibration candidates

### Scientific motivation

For photometric stability studies we need objects whose **flux in each band
is predictable** given only the phase $\phi = (t \mod P) / P$.  
The requirements are:

1. **Well-determined period** (low Lomb-Scargle FAP)
2. **Multi-band coverage** (≥ 2 LSST bands, ideally $g, r, i$)
3. **Smooth, repeatable light-curve shape** → low Fourier-fit residuals
4. **Period in a stable range** (not too long: Miras drift; not too short:
   delta Scuti are hard to phase-fold with LSST sparse cadence)

**Best candidates in practice:**
- **RR Lyrae (RRAB)**: P ≈ 0.3–1 d, sharp asymmetric light curve,
  well-modelled by order-4 Fourier series, extensively used for
  photometric cross-calibration (e.g. Pan-STARRS PS1 calibration).
- **Classical Cepheids (DCEP)**: P ≈ 1–100 d, smoother,
  standard candles — ideal if found in galactic fields.
- **W Vir / Type II Cep**: P ≈ 2–30 d, less abundant.

### Method

For each selected object and each band, we fit:

$$f(\phi) = A_0 + \sum_{k=1}^{N_{\rm order}} \left[ A_k \cos(2\pi k \phi) + B_k \sin(2\pi k \phi) \right]$$

with $N_{\rm order} = 4$ (adjustable).  
Quality metric: **reduced $\chi^2_\nu$** and **RMS residual in mmag**.
Objects with $\chi^2_\nu < 3$ and RMS $< 50$ mmag are flagged as
`phot_cal_candidate = True`.


In [ ]:
# ── Section 14a: selection criteria ──────────────────────────────────────────
import numpy.linalg as npl

# Tuneable thresholds
FAP_MAX = 0.05  # LS false-alarm probability cut
PERIOD_MIN_CAL = 0.25  # days — exclude sub-day noise
PERIOD_MAX_CAL = 150.0  # days — exclude slow Miras
N_SRC_MIN_BAND = 8  # minimum detections per band for a fit
FOURIER_ORDER = 4  # truncation order for the Fourier series
CHI2_MAX = 5.0  # reduced chi2 threshold for "good fit"
RMS_MAX_MMAG = 80.0  # RMS residual threshold (mmag)

# Priority classes for photometric calibration
CALIB_CLASSES = ["rr_lyrae", "cepheid_classical", "cepheid_type2", "delta_scuti"]


def fourier_design_matrix(phi: np.ndarray, order: int) -> np.ndarray:
    """Build the Fourier design matrix for phase array phi in [0,1)."""
    cols = [np.ones(len(phi))]
    for k in range(1, order + 1):
        cols.append(np.cos(2 * np.pi * k * phi))
        cols.append(np.sin(2 * np.pi * k * phi))
    return np.column_stack(cols)  # shape (N, 2*order+1)


def fit_fourier(phi: np.ndarray, mag: np.ndarray, mag_err: np.ndarray, order: int = FOURIER_ORDER):
    """
    Weighted least-squares Fourier fit in phase space.

    Returns
    -------
    coeffs   : array (2*order+1,)
    mag_pred : array (N,)  — model evaluated at input phases
    chi2_nu  : float       — reduced chi2
    rms_mmag : float       — RMS of (mag - model) in mmag
    """
    W = np.diag(1.0 / np.maximum(mag_err, 1e-6) ** 2)
    A = fourier_design_matrix(phi, order)
    # Weighted normal equations:  (A^T W A) c = A^T W mag
    AtW = A.T @ W
    AtWA = AtW @ A
    AtWy = AtW @ mag
    try:
        coeffs = npl.solve(AtWA, AtWy)
    except npl.LinAlgError:
        coeffs = npl.lstsq(A, mag, rcond=None)[0]
    mag_pred = A @ coeffs
    residuals = mag - mag_pred
    ndof = max(1, len(mag) - (2 * order + 1))
    chi2_nu = float(np.sum((residuals / np.maximum(mag_err, 1e-6)) ** 2) / ndof)
    rms_mmag = float(np.std(residuals) * 1000.0)
    return coeffs, mag_pred, chi2_nu, rms_mmag


def eval_fourier(phi_dense: np.ndarray, coeffs: np.ndarray, order: int = FOURIER_ORDER) -> np.ndarray:
    """Evaluate Fourier model on a dense phase grid."""
    A = fourier_design_matrix(phi_dense, order)
    return A @ coeffs


# ── Apply selection ───────────────────────────────────────────────────────────
if df_periods.empty:
    print("No period data — skipping calibration candidate selection.")
    df_calib = pd.DataFrame()
else:
    mask_fap = df_periods["ls_fap"] <= FAP_MAX
    mask_period = (df_periods["best_period_d"] >= PERIOD_MIN_CAL) & (
        df_periods["best_period_d"] <= PERIOD_MAX_CAL
    )
    mask_class = df_periods["pulsator_class"].isin(CALIB_CLASSES)

    df_calib = df_periods[mask_fap & mask_period & mask_class].copy()
    print(f"Selection (FAP≤{FAP_MAX}, P in [{PERIOD_MIN_CAL},{PERIOD_MAX_CAL}]d, class in {CALIB_CLASSES}):")
    print(f"  {len(df_calib)} candidates out of {len(df_periods)} total.")
    if not df_calib.empty:
        display(
            df_calib[["diaObjectId", "pulsator_class", "best_period_d", "ls_fap", "n_pts"]].sort_values(
                "ls_fap"
            )
        )

In [ ]:
# ── Section 14b: Fourier fitting per band ────────────────────────────────────
fit_results = []  # one dict per (object, band)
fit_store = {}  # lc_dict-like: {oid: {band: {coeffs, phi, mag, mag_pred, ...}}}

if df_calib.empty:
    print("No calibration candidates — Fourier fitting skipped.")
else:
    for _, row in df_calib.iterrows():
        oid = row["diaObjectId"]
        period = row["best_period_d"]
        data = lc_dict.get(oid, {})
        if not data or data["src"].empty:
            continue
        df_filt = filter_lc(data["src"])
        if df_filt.empty:
            continue

        t0 = df_filt["r:midpointMjdTai"].min()
        fit_store[oid] = {}

        for band in BANDS:
            dfb = df_filt[df_filt["r:band"] == band]
            if len(dfb) < N_SRC_MIN_BAND:
                continue
            phi = phase_fold(dfb["r:midpointMjdTai"].values, period, t0)
            mag = dfb["mag"].values
            mag_err = dfb["mag_err"].values

            coeffs, mag_pred, chi2_nu, rms_mmag = fit_fourier(phi, mag, mag_err, order=FOURIER_ORDER)
            fit_store[oid][band] = {
                "phi": phi,
                "mag": mag,
                "mag_err": mag_err,
                "mag_pred": mag_pred,
                "coeffs": coeffs,
                "chi2_nu": chi2_nu,
                "rms_mmag": rms_mmag,
                "n_pts": len(dfb),
            }
            fit_results.append(
                {
                    "diaObjectId": oid,
                    "band": band,
                    "pulsator_class": row["pulsator_class"],
                    "period_d": period,
                    "n_pts": len(dfb),
                    "chi2_nu": chi2_nu,
                    "rms_mmag": rms_mmag,
                    "phot_cal_ok": (chi2_nu <= CHI2_MAX) and (rms_mmag <= RMS_MAX_MMAG),
                }
            )
            print(
                f"  {oid}  {band}  chi2_nu={chi2_nu:.2f}  RMS={rms_mmag:.1f} mmag  "
                f"({'OK' if (chi2_nu <= CHI2_MAX and rms_mmag <= RMS_MAX_MMAG) else 'FAIL'})"
            )

    df_fits = pd.DataFrame(fit_results)
    if not df_fits.empty:
        df_fits.to_parquet(DIR_DATA / "fourier_fit_quality.parquet", index=False)
        print(f"\nSaved fourier_fit_quality.parquet ({len(df_fits)} rows).")
        display(df_fits.sort_values(["chi2_nu", "band"]))

In [ ]:
# ── Section 14c: plot phase-folded data + Fourier model per object ────────────
phi_dense = np.linspace(0, 1, 500)

if not fit_store:
    print("No Fourier fits to plot.")
else:
    for oid, band_fits in fit_store.items():
        if not band_fits:
            continue
        bands_fitted = sorted(band_fits.keys())
        n_bands = len(bands_fitted)
        meta = lc_dict[oid]["meta"]
        period = df_calib[df_calib["diaObjectId"] == oid]["best_period_d"].values[0]

        ncols = min(3, n_bands)
        nrows = int(np.ceil(n_bands / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
        axes = axes.flatten()

        for k, band in enumerate(bands_fitted):
            d = band_fits[band]
            ax = axes[k]
            col = BAND_COLORS.get(band, "grey")

            # Data (duplicated over 2 periods)
            phi_2 = np.concatenate([d["phi"], d["phi"] + 1])
            mag_2 = np.concatenate([d["mag"], d["mag"]])
            err_2 = np.concatenate([d["mag_err"], d["mag_err"]])
            pred_2 = np.concatenate([d["mag_pred"], d["mag_pred"]])

            ax.errorbar(phi_2, mag_2, yerr=err_2, fmt="o", ms=3, lw=0.4, color=col, alpha=0.7, label="data")

            # Fourier model curve (smooth)
            model_dense = eval_fourier(
                np.concatenate([phi_dense, phi_dense + 1]),
                d["coeffs"],
            )
            ax.plot(
                np.concatenate([phi_dense, phi_dense + 1]),
                model_dense,
                color="black",
                lw=1.5,
                zorder=5,
                label=f"Fourier O={FOURIER_ORDER}",
            )

            ax.set_xlim(0, 2)
            ax.invert_yaxis()
            ax.set_xlabel("Phase (× 2 periods)")
            ax.set_ylabel("AB mag")
            ax.set_title(
                f"band={band}  χ²ν={d['chi2_nu']:.2f}  RMS={d['rms_mmag']:.1f} mmag\n"
                f"N={d['n_pts']}  P={period:.3f}d",
                fontsize=8,
            )
            ax.legend(fontsize=7)

        for ax in axes[n_bands:]:
            ax.set_visible(False)

        pclass = meta.get("pulsator_class", "?")
        plt.suptitle(
            f"{oid} — {pclass} | P={period:.3f}d  (Fourier order {FOURIER_ORDER})",
            fontsize=10,
            y=1.01,
        )
        plt.tight_layout()
        savefig(f"fourier_fit_{oid}")
        plt.show()

In [ ]:
# ── Section 14d: residual histogram per band (all candidates combined) ────────
if "df_fits" not in dir() or df_fits.empty:
    print("No fit results available.")
else:
    bands_in_fits = sorted(df_fits["band"].unique())
    ncols = min(3, len(bands_in_fits))
    nrows = int(np.ceil(len(bands_in_fits) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
    axes = axes.flatten()

    for k, band in enumerate(bands_in_fits):
        ax = axes[k]
        col = BAND_COLORS.get(band, "grey")

        # Collect all residuals (observed - Fourier model) in this band
        all_residuals_mmag = []
        for oid, band_fits in fit_store.items():
            if band not in band_fits:
                continue
            d = band_fits[band]
            resid = (d["mag"] - d["mag_pred"]) * 1000.0  # → mmag
            all_residuals_mmag.extend(resid.tolist())

        if not all_residuals_mmag:
            ax.set_visible(False)
            continue

        resid_arr = np.array(all_residuals_mmag)
        rms = np.std(resid_arr)
        ax.hist(resid_arr, bins=30, color=col, alpha=0.75, edgecolor="white")
        ax.axvline(0, color="black", lw=1.2, ls="-")
        ax.axvline(rms, color="red", lw=1.0, ls="--", label=f"+1σ={rms:.1f} mmag")
        ax.axvline(-rms, color="red", lw=1.0, ls="--", label=f"-1σ")
        ax.set_xlabel("Residual (obs − model)  [mmag]", fontsize=9)
        ax.set_ylabel("Count", fontsize=9)
        ax.set_title(f"Band {band}  —  N={len(resid_arr)}  RMS={rms:.1f} mmag", fontsize=9)
        ax.legend(fontsize=7)

    for ax in axes[len(bands_in_fits) :]:
        ax.set_visible(False)

    plt.suptitle(
        "Fourier-fit residuals per band — all photometric calibration candidates",
        fontsize=11,
        y=1.01,
    )
    plt.tight_layout()
    savefig("fourier_residuals_per_band")
    plt.show()

    # ── Final quality summary ────────────────────────────────────────────────
    print("\nPhotometric calibration candidate summary:")
    print("=" * 65)
    df_ok = df_fits[df_fits["phot_cal_ok"]]
    print(
        f"  Objects with ≥1 band passing (chi2ν<{CHI2_MAX}, RMS<{RMS_MAX_MMAG} mmag):"
        f" {df_ok['diaObjectId'].nunique()}"
    )
    if not df_ok.empty:
        print("\n  Best objects (lowest median chi2ν across bands):")
        best = (
            df_ok.groupby("diaObjectId")
            .agg(
                median_chi2=("chi2_nu", "median"),
                n_bands=("band", "count"),
                pulsator_class=("pulsator_class", "first"),
                period_d=("period_d", "first"),
            )
            .sort_values("median_chi2")
            .reset_index()
        )
        display(best)
    print("\nThese objects are your photometric-calibration candidates.")
    print("Their Fourier coefficients (saved above) can predict flux at any")
    print("future epoch given only the observation MJD and the fitted period.")